# MCP Clients

The **MCP client** is the bridge between your server and MCP servers — your access point to the tools a server provides. It handles all the message passing and protocol details so you don't have to.

> Concept lesson — no code. We implement a real client later at **Implementing a client**.

## Transport agnostic

The client and server can talk over different **transports**. The most common local setup runs both on the same machine, communicating over **standard input/output (stdio)**. They can also connect over the network:

- HTTP
- WebSockets
- other network protocols

> **Modern note:** today's two standard transports are **stdio** (local subprocess — what this section uses) and **Streamable HTTP** (remote), which superseded the older HTTP+SSE transport. Either way the *messages* below are identical — transport is just the pipe.

## Message types

Once connected, client and server exchange typed messages from the MCP spec. The two you'll use most:

| Request → Result | Meaning |
|------------------|---------|
| `ListToolsRequest` → `ListToolsResult` | "What tools do you provide?" → the list of available tools |
| `CallToolRequest` → `CallToolResult` | "Run this tool with these args" → the tool's output |

(There are parallel message types for resources and prompts, covered later.)

## The complete flow

User asks *"What repositories do I have?"* — here's every hop:

```
1. User → your server:      the query
2. your server → MCP client: "what tools exist?"
3. MCP client → MCP server:  ListToolsRequest  →  ListToolsResult (tool list)
4. your server → Claude:      user question + available tools
5. Claude → your server:      tool_use request (it picked a tool)
6. your server → MCP client:  "run that tool"
7. MCP client → MCP server:   CallToolRequest
8. MCP server → GitHub:       the real API call
9. GitHub → MCP server → MCP client → your server:  CallToolResult (repo data)
10. your server → Claude:      tool_result (follow-up message)
11. Claude → your server → user:  the formatted answer
```

Many steps, but each component has one clear job. The **MCP client abstracts away the server communication** (steps 3, 7, 9) so your app logic stays focused: get tools, call Claude, execute the tool Claude wants, return results.

Notice this is the **same request → tool_use → execute → tool_result loop** from the tool-use section — MCP just relocates *where the tools live and run* (the MCP server) and hands you a client to reach them.

Next: **Project setup** — scaffolding the server + client project (this is where code begins).